# Global Analysis of All Reconstructed Matrices

In [17]:
from pathlib import Path
import warnings
import networkx as nx
import json
import numpy as np
import pandas as pd
import torch

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


# Step 1: Load Full Reconstructed Dataset (Local PC)
Load reconstructed matrices.

In [18]:
WINDOW_LENGTH = 724
STRIDE = 10
DATASET_NAME = f'data_00_20_w{WINDOW_LENGTH}_s{STRIDE}'
RUN_NAME = 'AE_12dim_05'
FILE_NAME = f'all_reconstructed_{RUN_NAME}.pt'

project_root = Path.cwd().resolve().parent
base_dir = project_root / 'data' / 'processed' / 'dataset'
dataset_dir = base_dir / DATASET_NAME

ALL_RECONSTRUCTED_MATRIX_FILE = dataset_dir / FILE_NAME
RESULTS_FOLDER = project_root / 'results' / DATASET_NAME / 'reconstruction_analysis' / RUN_NAME
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

if not ALL_RECONSTRUCTED_MATRIX_FILE.exists():
    raise FileNotFoundError(
        f"File '{ALL_RECONSTRUCTED_MATRIX_FILE.name}' not found in: {dataset_dir.absolute()}"
    )

print(f'Dataset selected: {DATASET_NAME}')
print(f'All Reconstructed Matrix: {ALL_RECONSTRUCTED_MATRIX_FILE.name}')

Dataset selected: data_00_20_w724_s10
All Reconstructed Matrix: all_reconstructed_AE_12dim_05.pt


In [19]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')
    corr_tensor = payload.get('corr_tensor', None)
    recon_tensor = payload.get('corr_tensor_reconstructed', None)
    indices = payload.get('indices', None)
    meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')
    if recon_tensor is None:
        raise KeyError('corr_tensor_reconstructed key not found in .pt file')
    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')
    if recon_tensor.ndim != 3 or recon_tensor.shape[1] != recon_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor_reconstructed: {recon_tensor.shape}')

    return corr_tensor.float(), recon_tensor.float(), indices, meta


gt_corr, recon_corr, indices, meta = load_corr_payload(ALL_RECONSTRUCTED_MATRIX_FILE)

print(f'Ground truth correlation tensor shape: {gt_corr.shape}')
print(f'Reconstructed correlation tensor shape: {recon_corr.shape}')
print(f'Indices: {indices[:10]}')
print(f'Metadata keys: {list(meta.keys())}')
print(gt_corr[:1,:5,:5])
print(recon_corr[:1,:5,:5])

num_windows, num_assets, _ = gt_corr.shape
print(f'\nNumber of windows: {num_windows}, Number of assets: {num_assets}')


Ground truth correlation tensor shape: torch.Size([412, 362, 362])
Reconstructed correlation tensor shape: torch.Size([412, 362, 362])
Indices: [49, 50, 51, 52, 53, 54, 55, 56, 57, 58]
Metadata keys: ['indices', 'split', 'meta', 'corr_tensor_reconstructed', 'tickers']
tensor([[[1.0000, 0.1305, 0.1530, 0.2544, 0.4250],
         [0.1305, 1.0000, 0.1224, 0.3038, 0.1635],
         [0.1530, 0.1224, 1.0000, 0.1499, 0.2088],
         [0.2544, 0.3038, 0.1499, 1.0000, 0.2217],
         [0.4250, 0.1635, 0.2088, 0.2217, 1.0000]]])
tensor([[[0.9959, 0.1363, 0.0975, 0.2264, 0.3569],
         [0.1439, 0.9965, 0.1190, 0.2701, 0.1792],
         [0.0981, 0.1263, 0.9965, 0.1280, 0.1879],
         [0.2406, 0.2756, 0.1369, 0.9962, 0.2149],
         [0.3580, 0.1795, 0.1824, 0.2061, 0.9964]]])

Number of windows: 412, Number of assets: 362


# Errors Statistics (MSE,MAE,Frobenius on full Reconstructed Dataset)

In [20]:
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    diff = original - reconstructed

    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [21]:
gt_corr_np = gt_corr.cpu().numpy()
recon_corr_np = recon_corr.cpu().numpy()

errors_df, errors_stats = reconstruction_errors(gt_corr_np, recon_corr_np)

print('Reconstruction error statistics (Full Dataset - no matrix sanitization):')
display(errors_stats)

Reconstruction error statistics (Full Dataset - no matrix sanitization):


,mean,std,min,median,max
MSE,0.000331,0.000246,0.000087,0.000270,0.003938
MAE,0.012875,0.003914,0.006900,0.012385,0.048815
Frobenius,6.343812,1.778390,3.367635,5.944633,22.717003


In [22]:
def mst_edge_set(mst) -> set:
    return {frozenset(edge) for edge in mst.edges()}

def degree_distribution(mst, n_assets: int) -> np.ndarray:
    degrees = np.array([deg for _, deg in mst.degree()], dtype=int)
    counts = np.bincount(degrees, minlength=n_assets)
    return counts / counts.sum()

def compare_mst_metrics(mst_orig, mst_recon, n_assets: int, top_k: int):
    # Edges overlap and Jaccard
    edges_orig = mst_edge_set(mst_orig)
    edges_recon = mst_edge_set(mst_recon)
    common_edges = len(edges_orig & edges_recon)
    total_edges = max(n_assets - 1, 1)
    edge_overlap_pct = 100.0 * common_edges / total_edges
    edge_jaccard_pct = 100.0 * common_edges / max(len(edges_orig | edges_recon), 1)

    # Degree distribution and L1 distance
    dist_orig = degree_distribution(mst_orig, n_assets)
    dist_recon = degree_distribution(mst_recon, n_assets)
    degree_l1 = float(np.sum(np.abs(dist_orig - dist_recon)))

    # Average path length (weighted by distance)
    avg_path_len = nx.average_shortest_path_length(mst_orig, weight='weight')
    avg_path_len_recon = nx.average_shortest_path_length(mst_recon, weight='weight')
    avg_path_len_diff = float(abs(avg_path_len - avg_path_len_recon))

    # Average path length (unweighted, treating all edges as length 1)
    avg_path_len_unw = nx.average_shortest_path_length(mst_orig)
    avg_path_len_unw_recon = nx.average_shortest_path_length(mst_recon)
    avg_path_len_unw_diff = float(abs(avg_path_len_unw - avg_path_len_unw_recon))

    # Betweenness centrality top-k overlap
    top_k = int(min(top_k, n_assets))
    bet_orig = nx.betweenness_centrality(mst_orig, weight='weight', normalized=True)
    bet_recon = nx.betweenness_centrality(mst_recon, weight='weight', normalized=True)
    top_orig = {n for n, _ in sorted(bet_orig.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    top_recon = {n for n, _ in sorted(bet_recon.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    bet_overlap_pct = 100.0 * len(top_orig & top_recon) / max(top_k, 1)

    return {
        'edge_overlap_pct': edge_overlap_pct,
        'edge_jaccard_pct': edge_jaccard_pct,
        'degree_l1': degree_l1,
        'avg_path_len': float(avg_path_len),
        'avg_path_len_recon': float(avg_path_len_recon),
        'avg_path_len_diff': avg_path_len_diff,
        'avg_path_len_unweighted': float(avg_path_len_unw),
        'avg_path_len_unweighted_recon': float(avg_path_len_unw_recon),
        'avg_path_len_unweighted_diff': avg_path_len_unw_diff,
        'betweenness_topk_overlap_pct': bet_overlap_pct,
    }

In [23]:
def sanitize_correlation_matrix(corr_matrix):
    corr_matrix = np.clip(corr_matrix, -1.0, 1.0)
    corr_matrix = (corr_matrix + corr_matrix.T) / 2.0
    np.fill_diagonal(corr_matrix, 1.0)
    
    return corr_matrix

def build_distance_matrix(corr_matrix):
    dist_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_matrix)))
    dist_matrix = (dist_matrix + dist_matrix.T) / 2.0
    np.fill_diagonal(dist_matrix, 0.0)
    
    return dist_matrix

def corr_to_mst(corr_matrix):
    """
    Converte una matrice di correlazione in un oggetto MST di NetworkX.
    Usa la metrica di distanza d = sqrt(2 * (1 - rho))
    """
    # 1. Calcolo della matrice delle distanze
    corr_matrix = sanitize_correlation_matrix(corr_matrix)
    dist_matrix = build_distance_matrix(corr_matrix)
    
    # 2. Creazione del grafo completo
    G = nx.from_numpy_array(dist_matrix)
    
    # 3. Calcolo del Minimum Spanning Tree
    mst = nx.minimum_spanning_tree(G, weight='weight')
    
    return mst

In [24]:
# Inizializziamo una lista per contenere i risultati di ogni coppia di matrici
all_metrics = []

n_samples = gt_corr_np.shape[0]
n_assets = gt_corr_np.shape[1]
top_k_centrality = 10

print(f"Inizio elaborazione di {n_samples} matrici...")

for i in range(n_samples):
    # 1. Estrazione della singola matrice (2D)
    curr_gt = gt_corr_np[i]
    curr_recon = recon_corr_np[i]
    
    # 2. Generazione degli MST per questa istanza
    mst_gt = corr_to_mst(curr_gt)
    mst_recon = corr_to_mst(curr_recon)
    
    # 3. Calcolo metriche
    metrics = compare_mst_metrics(
        mst_gt, 
        mst_recon, 
        n_assets=n_assets, 
        top_k=top_k_centrality
    )
    
    # Aggiungiamo alla lista
    all_metrics.append(metrics)
    
    # Feedback ogni 50 iterazioni per monitorare il progresso
    if (i + 1) % 50 == 0:
        print(f"Elaborate {i + 1}/{n_samples} matrici...")

# 4. Creazione DataFrame e calcolo della media
results_df = pd.DataFrame(all_metrics)

# 5. Calcolo delle statistiche aggregate
stats_df = results_df.describe().T 

# 6. Conversione in un dizionario strutturato
# 'index' orient crea un dizionario dove le chiavi sono le metriche
stats_dict = stats_df.to_dict(orient='index')

# 7. Salvataggio in formato JSON
file_path = RESULTS_FOLDER / 'mst_comparison_metrics.json'

with open(file_path, 'w') as f:
    json.dump(stats_dict, f, indent=4)

print(f"Statistiche salvate con successo in: {file_path}")

# --- Visualizzazione rapida a schermo delle statistiche aggregate ---
print("\n--- Summary Statistics (Full Dataset) ---")
display(stats_df)

Inizio elaborazione di 412 matrici...
Elaborate 50/412 matrici...
Elaborate 100/412 matrici...
Elaborate 150/412 matrici...
Elaborate 200/412 matrici...
Elaborate 250/412 matrici...
Elaborate 300/412 matrici...
Elaborate 350/412 matrici...
Elaborate 400/412 matrici...
Statistiche salvate con successo in: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\results\data_00_20_w724_s10\reconstruction_analysis\AE_12dim_05\mst_comparison_metrics.json

--- Summary Statistics (Full Dataset) ---


,count,mean,std,min,25%,50%,75%,max
edge_overlap_pct,412.0,78.343598,5.809976,47.368421,74.515235,78.947368,82.825485,90.581717
edge_jaccard_pct,412.0,64.761636,7.663486,31.034483,59.381898,65.217391,70.685579,82.784810
degree_l1,412.0,0.093601,0.026644,0.033149,0.071823,0.093923,0.116022,0.176796
avg_path_len,412.0,7.834480,1.373753,4.925977,6.906827,7.625463,8.775517,11.573367
avg_path_len_recon,412.0,7.926464,1.302280,5.749979,6.770642,7.699148,8.839871,11.396664
avg_path_len_diff,412.0,0.586559,0.507180,0.004292,0.190519,0.459554,0.853247,2.796517
avg_path_len_unweighted,412.0,10.318442,1.493487,7.032828,9.224316,10.110421,11.442984,14.683491
avg_path_len_unweighted_recon,412.0,10.413053,1.314652,8.405075,9.349474,10.282296,11.318904,14.228892
avg_path_len_unweighted_diff,412.0,0.874976,0.738675,0.000934,0.308673,0.677484,1.274847,3.819746
betweenness_topk_overlap_pct,412.0,64.538835,17.579322,0.000000,50.000000,60.000000,80.000000,100.000000
